# Project 2: food desert community compared to a non-food desert community

### Author:

In [1]:
## package imports and downloads

%pip install eep153_tools
%pip install python_gnupg
%pip install -U gspread_pandas

import pandas as pd
from eep153_tools.sheets import read_sheets
from  scipy.optimize import linprog as lp
import numpy as np

  Using cached eep153_tools-0.12.4-py2.py3-none-any.whl.metadata (363 bytes)
Using cached eep153_tools-0.12.4-py2.py3-none-any.whl (4.9 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached python_gnupg-0.5.6-py2.py3-none-any.whl.metadata (2.1 kB)
Using cached python_gnupg-0.5.6-py2.py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached gspread_pandas-3.3.0-py2.py3-none-any.whl.metadata (10 kB)
Using cached gspread_pandas-3.3.0-py2.py3-none-any.whl (27 kB)
  Attempting uninstall: gspread_pandas
    Found existing installation: gspread-pandas 2.2.3
    Uninstalling gspread-pandas-2.2.3:
      Successfully uninstalled gspread-pandas-2.2.3
Note: you may need to restart the kernel to use updated packages.


In [2]:
RDI_SHEET_URL = "https://docs.google.com/spreadsheets/d/1y95IsQ4HKspPW3HHDtH7QMtlDA66IUsCHJLutVL-MMc/"

## dri function

def get_dri(sex: str, age: int) -> pd.Series:

    sex = sex.lower()
    
    if 1 <= age <= 3:
        age_group = "1-3"
        sex_letter = "C"

    else:
        sex_letter = "M" if sex == "male" else "F"
        
        if 4 <= age <= 8:
            age_group = "4-8"
        elif 9 <= age <= 13:
            age_group = "9-13"
        elif 14 <= age <= 18:
            age_group = "14-18"
        elif 19 <= age <= 30:
            age_group = "19-30"
        elif 31 <= age <= 50:
            age_group = "31-50"
        elif age >= 51:
            age_group = "51+"
        else:
            raise ValueError("This function currently supports ages 1+")


    column_name = f"{sex_letter} {age_group}"

    RDIs = read_sheets(RDI_SHEET_URL)
    diet_min = RDIs["diet_minimums"].set_index("Nutrition")

    return diet_min[column_name].astype(float)

In [3]:
## making sure it works
bmin_ex = get_dri("male", 62)
bmin_ex

Nutrition
Energy                            2000.0
Protein                             56.0
Fiber, total dietary                28.0
Folate, DFE                        400.0
Calcium, Ca                       1000.0
Carbohydrate, by difference        130.0
Iron, Fe                             8.0
Magnesium, Mg                      420.0
Niacin                              16.0
Phosphorus, P                      700.0
Potassium, K                      4700.0
Riboflavin                           1.3
Thiamin                              1.2
Vitamin A, RAE                     900.0
Vitamin B-12                         2.4
Vitamin B-6                          1.7
Vitamin C, total ascorbic acid      90.0
Vitamin E (alpha-tocopherol)        15.0
Vitamin K (phylloquinone)          120.0
Zinc, Zn                            11.0
Name: M 51+, dtype: float64

In [4]:
data_url = "https://docs.google.com/spreadsheets/d/12Z4n8HbFZRYvH6B-D8EDLDibRiL50zNMlSBLMJ41C1o/"

## helper function to standardize FNDDS codes

def format_id(id,zeropadding=0):

    if pd.isnull(id) or id in ['','.']: return None

    try:
        return ('%d' % id).zfill(zeropadding)
    except TypeError:
        return id.split('.')[0].strip().zfill(zeropadding)
    except ValueError:
        return None

## dataframe of recipes

recipes = read_sheets(data_url, sheet="recipes")
recipes = (recipes
           .assign(parent_foodcode = lambda df: df["parent_foodcode"].apply(format_id),
                   ingred_code = lambda df: df["ingred_code"].apply(format_id))
           .rename(columns={"parent_desc": "recipe"}))


recipes[recipes["recipe"].str.contains("Spaghetti", case=False)]

,parent_foodcode,recipe,ingred_code,ingred_desc,ingred_wt
1090,28133110,"Veal, breaded, with spaghetti, in tomato sauce...",1033,"Cheese, parmesan, hard",0.5
1091,28133110,"Veal, breaded, with spaghetti, in tomato sauce...",1038,"Cheese, romano",0.5
1092,28133110,"Veal, breaded, with spaghetti, in tomato sauce...",1129,"Egg, whole, cooked, hard-boiled",4.0
1093,28133110,"Veal, breaded, with spaghetti, in tomato sauce...",2047,"Salt, table",1.8
1094,28133110,"Veal, breaded, with spaghetti, in tomato sauce...",4034,"Oil, soybean, salad or cooking, (partially hyd...",8.7
...,...,...,...,...,...
32665,74404050,"Spaghetti sauce, reduced sodium",6976,"Sauce, pasta, spaghetti/marinara, ready-to-ser...",100.0
32666,74404060,"Spaghetti sauce, fat free",11549,"Tomato products, canned, sauce",100.0
33463,75233220,"Spaghetti squash, cooked",2047,"Salt, table, iodized",0.3
33464,75233220,"Spaghetti squash, cooked",4321,"Oil, table fat, averaged",3.0


In [5]:
## nutrient profiles for ingredients

ingredient_nutrients = (read_sheets(data_url, sheet="nutrients")
             .assign(ingred_code = lambda df: df["ingred_code"].apply(format_id)))

display(ingredient_nutrients.head())
ingredient_nutrients.columns

,ingred_code,Ingredient description,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,...,Vitamin B12,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc
0,1001,"Butter, salted",2.529,2.587,7.436,21.697,0.961,9.999,19.961,2.728,...,0.17,0.0,0.003,0.0,0.0,2.32,0.0,7.0,15.87,0.09
1,1002,"Butter, whipped, with salt",2.039,2.354,7.515,20.531,1.417,7.649,17.370,2.713,...,0.07,0.0,0.008,0.0,0.0,1.37,0.0,4.6,16.72,0.05
2,1003,"Butter oil, anhydrous",2.495,2.793,10.005,26.166,2.228,12.056,25.026,2.247,...,0.01,0.0,0.001,0.0,0.0,2.80,0.0,8.6,0.24,0.01
3,1004,"Cheese, blue",0.601,0.491,3.301,9.153,0.816,3.235,6.622,0.536,...,1.22,0.0,0.166,0.0,0.5,0.25,0.0,2.4,42.41,2.66
4,1005,"Cheese, brick",0.585,0.482,3.227,8.655,0.817,3.455,7.401,0.491,...,1.26,0.0,0.065,0.0,0.5,0.26,0.0,2.5,41.11,2.60


Index(['ingred_code', 'Ingredient description', 'Capric acid', 'Lauric acid',
       'Myristic acid', 'Palmitic acid', 'Palmitoleic acid', 'Stearic acid',
       'Oleic acid', 'Linoleic Acid', 'Linolenic Acid', 'Stearidonic acid',
       'Eicosenoic acid', 'Arachidonic acid', 'Eicosapentaenoic acid',
       'Erucic acid', 'Docosapentaenoic acid', 'Docosahexaenoic acid',
       'Butyric acid', 'Caproic acid', 'Caprylic acid', 'Alcohol', 'Caffeine',
       'Calcium', 'Carbohydrate', 'Carotene, alpha', 'Carotene, beta',
       'Cholesterol', 'Choline', 'Copper', 'Cryptoxanthin, beta', 'Energy',
       'Fatty acids, total monounsaturated',
       'Fatty acids, total polyunsaturated', 'Fatty acids, total saturated',
       'Dietary Fiber', 'Folate, DFE', 'Folate, food', 'Folate', 'Folic acid',
       'Iron', 'Lutein + zeaxanthin', 'Lycopene', 'Magnesium', 'Niacin',
       'Phosphorus', 'Potassium', 'Protein', 'Retinol', 'Riboflavin',
       'Selenium', 'Sodium', 'Sugars, total', 'Theobromin

In [6]:
## creating A matrix

recipes['ingred_wt'] = recipes['ingred_wt']/recipes.groupby(['parent_foodcode'])['ingred_wt'].transform("sum")

df = recipes.merge(ingredient_nutrients, how = "left", on = "ingred_code")

numeric_cols = list(df.select_dtypes(include = ["number"]).columns)
numeric_cols.remove("ingred_wt")
df[numeric_cols] = df[numeric_cols].mul(df["ingred_wt"], axis=0)

df = df.groupby('parent_foodcode').agg({**{col: "sum" for col in numeric_cols},
                                        "recipe": "first"})

df.index.name = "recipe_id"

food_names = df["recipe"]
print(food_names.head())
df.head()

recipe_id
11000000                       Milk, human
11100000                         Milk, NFS
11111000                       Milk, whole
11111100           Milk, low sodium, whole
11111150    Milk, calcium fortified, whole
Name: recipe, dtype: object


,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,Linolenic Acid,Stearidonic acid,...,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc,recipe
recipe_id,,,,,,,,,,,,,,,,,,,,,
11000000,0.06300,0.2560,0.32100,0.91900,0.12900,0.29300,1.475,0.3740,0.052,0.0,...,0.0,0.011,5.00,0.100,0.08,0.0,0.30,87.500,0.1700,"Milk, human"
11100000,0.03825,0.0405,0.14275,0.42475,0.01175,0.18575,0.397,0.0535,0.022,0.0,...,0.0,0.037,0.05,1.225,0.03,0.0,0.15,89.525,0.4225,"Milk, NFS"
11111000,0.07500,0.0770,0.29700,0.82900,0.00000,0.36500,0.812,0.1200,0.075,0.0,...,0.0,0.036,0.00,1.300,0.07,0.0,0.30,88.130,0.3700,"Milk, whole"
11111100,0.08700,0.0970,0.34800,0.91000,0.07700,0.41900,0.870,0.0780,0.050,0.0,...,0.0,0.034,0.90,1.300,0.08,0.0,0.30,88.200,0.3800,"Milk, low sodium, whole"
11111150,0.07500,0.0770,0.29700,0.82900,0.00000,0.36500,0.812,0.1200,0.075,0.0,...,0.0,0.036,0.00,1.300,0.07,0.0,0.30,88.130,0.3700,"Milk, calcium fortified, whole"


In [7]:
## creating p vector

prices = read_sheets(data_url, sheet = "prices")[["food_code", "year", "price"]]

prices["food_code"] = prices["food_code"].apply(format_id)

prices = prices.set_index(["year", "food_code"])
print(prices.index.levels[0])

prices = prices.xs("2017/2018", level = "year")

prices = prices.dropna(subset = "price")

Index(['2011/2012', '2013/2014', '2015/2016', '2017/2018'], dtype='object', name='year')


In [9]:
## loading nutrient requirements
## NOTE: different source than dri function above

rda = read_sheets(data_url, sheet = "rda")

rda = rda.set_index("Nutrient")

In [10]:
## matching nutritional content with foods and prices

common_recipes = df.index.intersection(prices.index)

df = df.loc[common_recipes]
prices = prices.loc[common_recipes]

prices.index = prices.index.map(food_names)

A_all = df.T

print(prices.head())
print(A_all.head())

                           price
Milk, NFS               0.100484
Milk, whole              0.09828
Milk, reduced fat (2%)  0.092085
Milk, low fat (1%)      0.090914
Milk, fat free (skim)   0.092441
                 11100000 11111000 11112110 11112210 11113000 11114300  \
Capric acid       0.03825    0.075    0.049    0.027    0.002    0.027   
Lauric acid        0.0405    0.077    0.055    0.029    0.001    0.029   
Myristic acid     0.14275    0.297    0.175    0.091    0.008    0.091   
Palmitic acid     0.42475    0.829    0.558    0.287    0.025    0.287   
Palmitoleic acid  0.01175      0.0    0.027    0.017    0.003    0.017   

                 11114330 11114350 11115000 11115100  ... 95312410 95312560  \
Capric acid         0.049    0.075    0.022    0.022  ...      0.0      0.0   
Lauric acid         0.055    0.077    0.025    0.025  ...      0.0      0.0   
Myristic acid       0.175    0.297    0.089    0.089  ...      0.0      0.0   
Palmitic acid       0.558    0.829    0.2

In [12]:
## building constraints

group = "Female_19_30"

bmin = rda.loc[rda['Constraint Type'].isin(['RDA', 'AI']), group]
bmax = rda.loc[rda['Constraint Type'].isin(['UL']), group]

Amin = A_all.reindex(bmin.index).dropna(how='all')
Amax = A_all.reindex(bmax.index).dropna(how='all')

b = pd.concat([bmin, -bmax])
A = pd.concat([Amin, -Amax])

#python tip: by typing "=" after the variable name inside the curly braces, it formats the output so we don't have to write f"variable = {variable}"
print(f"{bmin.shape=}")
print(f"{Amin.shape=}")
print(f"{bmax.shape=}")
print(f"{Amax.shape=}")
print(f"{b.shape=}")
print(f"{A.shape=}")
print(f"{prices.shape=}")

bmin.shape=(26,)
Amin.shape=(26, 4426)
bmax.shape=(1,)
Amax.shape=(1, 4426)
b.shape=(27,)
A.shape=(27, 4426)
prices.shape=(4426, 1)


In [15]:
## regression model

p = prices
tol = 1e-6
result = lp(p, -A, -b, method='highs')

In [16]:
print(f"Cost of diet for {group} is ${result.fun:.2f} per day.")

Cost of diet for Female_19_30 is $2.44 per day.


In [17]:
## subsistence diet

diet = pd.Series(result.x, index = prices.index)

print("\nYou'll be eating (in 100s of grams or milliliters):")
print(round(diet[diet >= tol], 2))


You'll be eating (in 100s of grams or milliliters):
Milk, low fat (1%)                                   8.41
Carp, steamed or poached                             0.06
Egg, yolk only, raw                                  0.09
Split peas, from dried, fat added                    4.84
Rice, white, cooked, made with margarine             1.08
Cereal (Kellogg's All-Bran Complete Wheat Flakes)    0.05
Cereal, toasted oat                                  0.19
Ripe plantain, raw                                   3.23
dtype: float64


In [19]:
## result

tab = pd.DataFrame({"Outcome":A.to_numpy()@diet.to_numpy(),"Recommendation":np.abs(b)})
print(tab)

                    Outcome  Recommendation
Nutrient                                   
Energy               2000.0          2000.0
Protein           76.714117            46.0
Carbohydrate     276.775494           130.0
Dietary Fiber     47.781799            28.0
Linoleic Acid      21.96157            12.0
Linolenic Acid     2.751783             1.1
Calcium         1219.941986          1000.0
Iron                   18.0            18.0
Magnesium        417.695502           310.0
Phosphorus      1568.466207           700.0
Potassium            4700.0          4700.0
Zinc              13.983042             8.0
Copper              1.31819             0.9
Selenium               55.0            55.0
Vitamin A       1277.799363           700.0
Vitamin E              15.0            15.0
Vitamin D              15.0            15.0
Vitamin C              75.0            75.0
Thiamin             1.81167             1.1
Riboflavin         2.606644             1.1
Niacin            14.599967     